# Session 37: Decision Trees and Random Forests

Data Science with Python, Level 3, Stage 6: Applied ML

## Learning Objectives

By the end of this session you will be able to:

1. Explain how a decision tree makes predictions.
2. Fit and draw a tree with scikit-learn.
3. Control overfitting with max_depth.
4. Explain how a random forest combines trees.
5. Read feature importances.
6. Compare trees, forests and logistic regression.

## Where We Are

Logistic regression adds up weighted features. A **decision tree** works the way people often decide: a series of yes or no questions. Is the contract monthly? Are the charges above 75? Each answer leads to the next question, and the last box gives the prediction.

A single tree overfits easily. A **random forest** trains hundreds of different trees and averages their votes, which is one of the most reliable models for tables. Today we compare both with the logistic regression churn model from Stage 5.

## Setup

We reuse the seven churn features and the same split as Sessions 33 and 34, so results are comparable.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor, plot_tree, export_text
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score

DATA_URL = "https://raw.githubusercontent.com/mohammadyahoo/ds-python-course/main/data/"
churn = pd.read_csv(DATA_URL + "telecom_churn.csv")
churn["left"] = (churn["churn"] == "Yes").astype(int)
churn["monthly_contract"] = (churn["contract"] == "Month-to-month").astype(int)
churn["fiber"] = (churn["internet"] == "Fiber").astype(int)
churn["echeck"] = (churn["payment_method"] == "Electronic check").astype(int)
features = ["tenure_months", "monthly_charges", "support_calls", "senior",
            "monthly_contract", "fiber", "echeck"]
X = churn[features].astype(float)
y = churn["left"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y)

## 1. Decision Trees

In [ ]:
tree = DecisionTreeClassifier(max_depth=2, random_state=0)
tree.fit(X_train, y_train)
print(export_text(tree, feature_names=features))

In [ ]:
plt.figure(figsize=(10, 5))
plot_tree(tree, feature_names=features, class_names=["stay", "leave"],
          filled=True, rounded=True, fontsize=9)
plt.show()

## 2. Trees Overfit Easily

In [ ]:
for depth in [1, 2, 4, 8, 12, None]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0)
    t.fit(X_train, y_train)
    train_acc = t.score(X_train, y_train)
    auc = roc_auc_score(y_test, t.predict_proba(X_test)[:, 1])
    print(depth, round(train_acc, 3), round(auc, 3), t.get_n_leaves())

## 3. Random Forests

In [ ]:
forest = RandomForestClassifier(n_estimators=300, min_samples_leaf=10,
                                random_state=0, n_jobs=-1)
forest.fit(X_train, y_train)

auc = roc_auc_score(y_test, forest.predict_proba(X_test)[:, 1])
print("Train accuracy:", round(forest.score(X_train, y_train), 3))
print("Test AUC:", round(auc, 3))

In [ ]:
for n in [1, 10, 50, 300]:
    f = RandomForestClassifier(n_estimators=n, min_samples_leaf=10,
                               random_state=0, n_jobs=-1).fit(X_train, y_train)
    print(n, "trees: AUC", round(roc_auc_score(y_test, f.predict_proba(X_test)[:, 1]), 3))

In [ ]:
importance = pd.Series(forest.feature_importances_, index=features)
importance.sort_values().plot.barh(color="tab:purple")
plt.title("Which features does the forest use most?")
plt.show()

In [ ]:
logreg = make_pipeline(StandardScaler(), LogisticRegression()).fit(X_train, y_train)
best_tree = DecisionTreeClassifier(max_depth=4, random_state=0).fit(X_train, y_train)
models = {"Tree": best_tree, "Forest": forest, "Logistic": logreg}
for name, m in models.items():
    auc = roc_auc_score(y_test, m.predict_proba(X_test)[:, 1])
    print(f"{name:9} AUC {auc:.3f}")

|  | Decision tree | Random forest | Logistic regression |
| --- | --- | --- | --- |
| Easy to explain | Very easy | Harder | Easy (odds ratios) |
| Needs scaling | No | No | Yes |
| Captures interactions | Yes | Yes | Only if added by hand |
| Overfitting risk | High | Low | Low |
| Training speed | Fast | Slower | Fast |

## Common Mistakes

| Symptom | Mistake | Fix |
| --- | --- | --- |
| No depth limit | Using a tree with default settings | Limit `max_depth` or `min_samples_leaf`. |
| Scaling trees | Adding StandardScaler before a tree | Trees do not need it; it is harmless but useless. |
| Importance as cause | Saying tenure causes churn because it is important | Importance shows use, not cause or direction. |
| Assuming forests win | Skipping the simple baseline model | Compare with logistic or linear regression. |

## In-Class Exercises

Work through these exercises during the session. Write your answer in the code cell under each one.

### In-Class Exercise 1: Read the Tree

Fit a depth 3 tree on the churn training data and print it with `export_text`. Describe in words the group with the highest predicted risk.

In [ ]:
# Write your code below

### In-Class Exercise 2: Depth Search

Use 5-fold cross-validated AUC on the training data to choose the best `max_depth` from 1 to 10 for a decision tree.

In [ ]:
# Write your code below

### In-Class Exercise 3: Forest for Prices

Load `apartments.csv` and compare the 5-fold cross-validated R2 of a depth 5 `DecisionTreeRegressor` and a `RandomForestRegressor` with 300 trees, using the numeric features and `floor`.

In [ ]:
# Write your code below

### In-Class Exercise 4: Leaf Size

Compare forests with `min_samples_leaf` of 1, 5, 10 and 30 on the churn data. Print training accuracy and test AUC for each.

In [ ]:
# Write your code below

## Summary

- A decision tree predicts with a sequence of yes or no questions.
- Trees need no scaling and capture interactions.
- Unlimited trees overfit; limit depth or leaf size.
- A random forest averages many randomised trees.
- Feature importances show which features a forest uses.
- Always compare with a simple baseline model.

## What's Next

A forest grows trees independently. Gradient boosting grows them one after another, each one fixing the errors of the previous ones. It powers many winning solutions on tabular data. Next session you will use it with scikit-learn and meet XGBoost.